# Choosing an optimizer

[`06a`](06a_variational_training.ipynb) trained a QAOA with Differential Evolution and [`06b`](06b_pso_qng_optimizers.ipynb) with PSO and QNG. `train()` takes more optimizers than those, all implemented in Rust and passed in the same `method` argument. This notebook trains the same MaxCut problem with the rest of them, shows how to compare them fairly, how to chain two of them, how to watch and stop a run, and how to plug in an optimizer written outside Polypus.

## The problem

The same graph and one-layer QAOA as in [`06a`](06a_variational_training.ipynb): three nodes, two edges, a best possible cut of 2, and two free parameters, `gamma` and `beta`. `common` gathers the arguments every run below shares, so each call only spells out what changes:

In [ ]:
import numpy as np
import polypus

Param = polypus.Param
edges = [(0, 1), (1, 2)]


def cut_value(bitstring):
    bits = bitstring[::-1]  # bits[i] = value of qubit i
    return sum(1 for i, j in edges if bits[i] != bits[j])


qc = polypus.Circuit(3)
qc.h(0).h(1).h(2)
qc.rzz(0, 1, Param(0))
qc.rzz(1, 2, Param(0))
qc.rx(0, Param(1)).rx(1, Param(1)).rx(2, Param(1))
qc.measure_all()

common = dict(
    shots=1000,
    n_qpus=1,
    expectation_function=cut_value,
    infrastructure="local",
    nodes=1,
    cores_per_qpu=2,
    seed=7,
)

## Five optimizers on the same problem

- **`DE`** is the Differential Evolution of [`06a`](06a_variational_training.ipynb). `strategy="shade"` and `strategy="lshade"` are adaptive variants that tune DE's own settings while they run; L-SHADE also shrinks the population as it goes. Classic DE also takes `mutation_factor` and `crossover_rate`, its `F` and `CR`; the adaptive variants tune both themselves, so they refuse them.
- **`CMAES`** also keeps a population, but samples it from a distribution that it moves and reshapes toward the better candidates.
- **`Adam`** follows the gradient of the cost, estimated from two extra circuits per parameter, with a step adapted to each parameter.
- **`SPSA`** also follows the gradient, but estimates all of it from just two circuits per iteration, by moving every parameter at once in a random direction.

There is a catch when comparing them: `best_fitness` is the best of many estimates measured with shot noise, so it tends to come out higher than the real value of its parameters, more so the more candidates an optimizer tries. `final_shots` measures the best candidates once more, after training, with many more shots, and reports the best of them as `final_fitness`, the fair figure to compare:

In [ ]:
optimizers = {
    "DE": polypus.DE(generations=30, population_size=10, seed=7),
    "L-SHADE": polypus.DE(
        strategy="lshade", generations=30, population_size=10, seed=7
    ),
    "CMA-ES": polypus.CMAES(generations=30, seed=7),
    "Adam": polypus.Adam(max_iters=30, seed=7),
    "SPSA": polypus.SPSA(max_iters=100, seed=7),
}
for name, optimizer in optimizers.items():
    result = polypus.train(
        qc,
        optimizer,
        dimensions=qc.num_params,
        id=f"maxcut-{name}",
        final_shots=20000,
        **common,
    )
    print(
        f"{name:>8}: best_fitness {result.best_fitness:.3f}  "
        f"final_fitness {result.final_fitness:.3f}  "
        f"iterations {result.iterations_run}"
    )

On a problem this small every optimizer reaches about the same cut, 1.65; as seen in [`06a`](06a_variational_training.ipynb), a single QAOA layer cannot reach 2 on this graph. The differences in `best_fitness` are shot noise. Real differences show up on bigger problems: the [training guide](https://github.com/Bahia-Software/polypus/blob/main/docs/guide/training.md#choosing-an-optimizer) compares them on graphs of 10 nodes with deeper circuits, at the same budget of circuit evaluations.

## Rotosolve: one parameter at a time, exactly

`Rotosolve` takes no step size. It moves one parameter at a time to the exact best value for it, computed from three circuits, which is only possible when each parameter is the angle of a single rotation. In this QAOA `Param(0)` is the angle of both `rzz` gates, so Rotosolve refuses the circuit before running anything:

In [ ]:
try:
    polypus.train(
        qc, polypus.Rotosolve(seed=7), dimensions=qc.num_params, id="roto", **common
    )
except ValueError as e:
    print(e)

A hardware-efficient ansatz does qualify: a layer of `ry` rotations with their own parameter each, a chain of `cx` and another layer of `ry`. Unlike the QAOA layer, this circuit can prepare the best partition itself, so Rotosolve reaches a cut close to the maximum of 2:

In [ ]:
hea = polypus.Circuit(3)
for q in range(3):
    hea.ry(q, Param(q))
hea.cx(0, 1).cx(1, 2)
for q in range(3):
    hea.ry(q, Param(3 + q))
hea.measure_all()

roto = polypus.train(
    hea,
    polypus.Rotosolve(sweeps=5, seed=7),
    dimensions=hea.num_params,
    id="roto-hea",
    **common,
)
print(
    "best fitness:",
    roto.best_fitness,
    "| iterations:",
    roto.iterations_run,
    "| converged:",
    roto.converged,
)

## Chaining two optimizers with `initial_point`

`initial_point` starts a run from given parameters instead of a random point. That makes it possible to chain two optimizers: one that explores the whole space, like L-SHADE, and then one that refines locally from its best point, like Adam. On the guide's benchmark, giving half the budget to each was the best of all the options:

In [ ]:
explore = polypus.train(
    qc,
    polypus.DE(strategy="lshade", generations=15, population_size=10, seed=7),
    dimensions=qc.num_params,
    id="explore",
    **common,
)
refine = polypus.train(
    qc,
    polypus.Adam(max_iters=15, seed=7),
    dimensions=qc.num_params,
    id="refine",
    initial_point=explore.best_params,
    **common,
)
print(f"L-SHADE:   {explore.best_fitness:.3f}")
print(f"then Adam: {refine.best_fitness:.3f}")

On a problem this small L-SHADE already reaches the best this one-layer QAOA can give, so Adam barely moves it; the gain of chaining them shows on bigger problems, like the guide's.

## Watching and stopping a run

`callback` is a function that `train()` calls after every iteration with a `TrainProgress`: the iteration, the best fitness and parameters so far, and how many candidates have been evaluated. It can print the progress, save a checkpoint or, by returning `True`, stop the run, here as soon as the mean cut reaches 1.4. A stopped run returns its result as usual, with `converged=False`:

In [ ]:
def report(progress):
    if progress.iteration % 5 == 0:
        print(
            f"iteration {progress.iteration:>2}: best {progress.best_fitness:.3f}, "
            f"{progress.evaluations} evaluations"
        )
    return progress.best_fitness >= 1.4


watched = polypus.train(
    qc,
    polypus.Adam(max_iters=60, seed=7),
    dimensions=qc.num_params,
    id="watched",
    callback=report,
    **common,
)
print("stopped at iteration", watched.iterations_run, "| converged:", watched.converged)

## An optimizer from outside Polypus: `External`

`polypus.External` runs any optimizer that works in ask/tell form, through the same scheduler as the others: `ask()` returns a batch of candidates, Polypus evaluates the batch, spread over the QPUs, and `tell(candidates, values)` gives back their fitness, which is always maximized. `max_evaluations` caps the candidates evaluated. Here the optimizer is a small local search written by hand:

In [ ]:
# samples a batch around the best point so far and moves there if it improves
class LocalSearch:
    def __init__(self, dimensions, batch_size, step, seed):
        self.rng = np.random.default_rng(seed)
        self.best = self.rng.uniform(0, np.pi, dimensions)
        self.best_value = -np.inf
        self.batch_size = batch_size
        self.step = step

    def ask(self):
        noise = self.rng.standard_normal((self.batch_size, len(self.best)))
        return self.best + self.step * noise

    def tell(self, candidates, values):
        i = int(np.argmax(values))
        if values[i] > self.best_value:
            self.best, self.best_value = np.array(candidates[i]), values[i]


external = polypus.train(
    qc,
    polypus.External(
        LocalSearch(2, batch_size=8, step=0.3, seed=7), max_evaluations=200
    ),
    dimensions=qc.num_params,
    id="external",
    **common,
)
print(
    "best fitness:",
    external.best_fitness,
    "| iterations:",
    external.iterations_run,
    "| converged:",
    external.converged,
)

`polypus.ask_tell` has ready-made adapters for Optuna, Nevergrad and pycma, `from_optuna`, `from_nevergrad` and `from_pycma`, which need the extras `[optuna]`, `[nevergrad]` and `[cma]`, as in `pip install "polypus-quantum[optuna]"`; the [training guide](https://github.com/Bahia-Software/polypus/blob/main/docs/guide/training.md#external-optimizers) shows each one.

## Summary

This notebook trained the same QAOA with L-SHADE, CMA-ES, Adam and SPSA, compared them on `final_fitness` instead of the noisy `best_fitness`, trained a circuit that qualifies for Rotosolve, chained L-SHADE and Adam with `initial_point`, watched and stopped a run with `callback`, and plugged in an optimizer of its own with `External`.

The next section trains models on data with `polypus.qml`, the native quantum machine learning module.

## Next step

Continue with: [`07a_native_classifier.ipynb`](07a_native_classifier.ipynb).